## Task 4: Tversky Loss

**Goal:** Use Tversky Loss as an alternative to cross-entropy on the same imbalanced dataset. Tversky Loss allows asymmetric penalisation of false positives and false negatives.

1. Implement `TverskyLoss`:

```python
class TverskyLoss(nn.Module):
    """
    TI   = (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    Loss = 1 - TI

    Special cases:
      alpha=0.5, beta=0.5  →  Dice Loss
      alpha=0.0, beta=1.0  →  pure Recall loss

    For imbalanced data, beta > alpha penalises FN more heavily → higher recall.
    """
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6, from_logits=True):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.smooth = smooth
        self.from_logits = from_logits

    def forward(self, logits, targets):
        if self.from_logits:
            probs = torch.sigmoid(logits.squeeze(1) if logits.dim() == 2 else logits)
        else:
            probs = logits.squeeze(1)
        targets = targets.float()
        tp = (probs * targets).sum()
        fp = (probs * (1.0 - targets)).sum()
        fn = ((1.0 - probs) * targets).sum()
        tversky_index = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return 1.0 - tversky_index
```

2. Train and evaluate analogously to Task 2, for several `(alpha, beta)` configurations:

| alpha | beta | effect                         |
|-------|------|--------------------------------|
| 0.5   | 0.5  | Dice Loss (symmetric)          |
| 0.3   | 0.7  | recall-biased                  |
| 0.1   | 0.9  | strong recall (penalise FN)    |

**Assignment:** Implement and train a classifier on a strongly imbalanced dataset — compare classification quality for BCE and Tversky Loss.


In [4]:
IMBALANCE_RATIO = 0.95

BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

In [5]:
from utils import make_imbalanced_dataset, BinaryClassifierMLP, make_loaders
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = BinaryClassifierMLP().to(device)

X, y = make_imbalanced_dataset(imbalance_ratio=IMBALANCE_RATIO)
print(f"Class 0: {(y == 0).sum():.0f}  Class 1: {(y == 1).sum():.0f}")

train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE, squeeze_y=True)

Class 0: 11288  Class 1: 712


In [6]:
from utils import train_baseline, TverskyLoss
import torch.nn as nn

TVERSKY_CONFIGS = [
    (0.5, 0.5),  # Dice Loss (symmetric)
    (0.3, 0.7),  # recall-biased
    (0.1, 0.9),  # strong recall (penalise FN)
]

print("Training with BCE: ")
model_bce = train_baseline(BinaryClassifierMLP().to(device), train_loader,
                           val_loader,criterion=nn.BCEWithLogitsLoss(),device=device,
                           weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                           )

tversky_models = {}
for alpha, beta in TVERSKY_CONFIGS:
    print(f"\nTraining with TverskyLoss(alpha={alpha}, beta={beta}): ")
    tversky_models[f"Tversky(α={alpha},β={beta})"] = (
        train_baseline(BinaryClassifierMLP().to(device), train_loader,
                    val_loader, criterion=TverskyLoss(alpha=alpha, beta=beta),
                    device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                       ))

Training with BCE...
Epoch 100/1000  train=0.0416  val=0.0968
Epoch 200/1000  train=0.0266  val=0.1296
Epoch 300/1000  train=0.0206  val=0.1358
Epoch 400/1000  train=0.0154  val=0.1462
Epoch 500/1000  train=0.0105  val=0.1591
Epoch 600/1000  train=0.0102  val=0.1893
Epoch 700/1000  train=0.0108  val=0.1855
Epoch 800/1000  train=0.0104  val=0.2046
Epoch 900/1000  train=0.0117  val=0.2017
Epoch 1000/1000  train=0.0099  val=0.2061

Training with TverskyLoss(alpha=0.5, beta=0.5)...
Epoch 100/1000  train=0.1519  val=0.2074
Epoch 200/1000  train=0.1442  val=0.2073
Epoch 300/1000  train=0.1299  val=0.1872
Epoch 400/1000  train=0.1231  val=0.2009
Epoch 500/1000  train=0.1134  val=0.1933
Epoch 600/1000  train=0.1090  val=0.1965
Epoch 700/1000  train=0.1120  val=0.1874
Epoch 800/1000  train=0.1051  val=0.1989
Epoch 900/1000  train=0.1195  val=0.1808
Epoch 1000/1000  train=0.1076  val=0.1852

Training with TverskyLoss(alpha=0.3, beta=0.7)...
Epoch 100/1000  train=0.1805  val=0.2478
Epoch 200/1000

In [7]:
from utils import get_probs, compute_clf_metrics

y_true, probs_bce = get_probs(model_bce, test_loader, device=device)
m_bce = compute_clf_metrics(y_true, probs_bce)

results = {"BCE": m_bce}
for name, model in tversky_models.items():
    y_true, probs = get_probs(model, test_loader, device=device)
    results[name] = compute_clf_metrics(y_true, probs)

print(f"\n{'Metric':<12}", end="")
for name in results:
    print(f" {name:>22}", end="")
print()
print("=" * (12 + 23 * len(results)))
for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{key:<12}", end="")
    for m in results.values():
        print(f" {m[key]:>22.4f}", end="")
    print()
print("=" * (12 + 23 * len(results)))
print()
for name, m in results.items():
    print(f"{name} — TP={m['tp']}  FP={m['fp']}  TN={m['tn']}  FN={m['fn']}")


Metric                          BCE   Tversky(α=0.5,β=0.5)   Tversky(α=0.3,β=0.7)   Tversky(α=0.1,β=0.9)
accuracy                     0.9738                 0.9762                 0.9704                 0.9667
precision                    0.8857                 0.9065                 0.7967                 0.7286
recall                       0.6458                 0.6736                 0.6806                 0.7083
f1                           0.7470                 0.7729                 0.7341                 0.7183
roc_auc                      0.9070                 0.9186                 0.9096                 0.9086
pr_auc                       0.7796                 0.7947                 0.7637                 0.7467

BCE — TP=93  FP=12  TN=2244  FN=51
Tversky(α=0.5,β=0.5) — TP=97  FP=10  TN=2246  FN=47
Tversky(α=0.3,β=0.7) — TP=98  FP=25  TN=2231  FN=46
Tversky(α=0.1,β=0.9) — TP=102  FP=38  TN=2218  FN=42
